# Step 4: Object Storage with MinIO

So far the "lake" has been a folder on local disk. Real data lakes live on **object storage** (S3, Azure Blob, GCS, ...): storage and compute are separate services, scale independently, and can be shared by many engines at once.

[MinIO](https://min.io) is an S3-compatible object store we can run as a container right inside the Codespace — same API as AWS S3, zero cloud account needed.

**Before running this notebook**, start MinIO from a terminal:

```bash
bash scripts/start_minio.sh
```

Codespaces will forward port **9001** automatically — open the "Ports" tab and click the link to see the MinIO web console (login: `minioadmin` / `minioadmin`).

In [ ]:
import os
from pathlib import Path

while not (Path.cwd() / "requirements.txt").exists():
    os.chdir("..")
print("Working directory:", Path.cwd())

## Create a bucket and upload the Parquet files

A bucket is the S3 equivalent of a top-level folder. We upload the partitioned Parquet files from `lake/verkauf/` into a bucket called `datalake`, keeping the same `jahr=.../monat=.../data.parquet` key structure — partitioning works identically on object storage.

In [ ]:
from minio import Minio

MINIO_ENDPOINT = "localhost:9000"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin"
BUCKET = "datalake"

client = Minio(
    MINIO_ENDPOINT,
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=False,
)

if not client.bucket_exists(BUCKET):
    client.make_bucket(BUCKET)
    print(f"Created bucket '{BUCKET}'")
else:
    print(f"Bucket '{BUCKET}' already exists")

In [ ]:
local_root = Path("lake/verkauf")
uploaded = 0

for file in local_root.rglob("*.parquet"):
    object_key = "verkauf/" + file.relative_to(local_root).as_posix()
    client.fput_object(BUCKET, object_key, str(file))
    uploaded += 1

print(f"Uploaded {uploaded} files to s3://{BUCKET}/verkauf/")

Open the MinIO console (port 9001) now — you should see the `datalake` bucket with the same `jahr=.../monat=.../` folder structure, visually, in the browser.

## Query MinIO directly from DuckDB

DuckDB's `httpfs` extension speaks the S3 API. `CREATE SECRET` registers the MinIO credentials and endpoint once per session; after that, `s3://...` paths work exactly like local paths.

In [ ]:
import duckdb

con = duckdb.connect()
con.sql("INSTALL httpfs")
con.sql("LOAD httpfs")

con.sql(f"""
    CREATE SECRET minio_secret (
        TYPE S3,
        KEY_ID '{MINIO_ACCESS_KEY}',
        SECRET '{MINIO_SECRET_KEY}',
        ENDPOINT '{MINIO_ENDPOINT}',
        URL_STYLE 'path',
        USE_SSL false,
        REGION 'us-east-1'
    )
""")
print("Secret registered.")

In [ ]:
con.sql("""
    SELECT region, SUM(revenue) AS total_revenue
    FROM 's3://datalake/verkauf/**/*.parquet'
    WHERE jahr = 2026
    GROUP BY region
    ORDER BY total_revenue DESC
""").show()

## The point of this whole notebook

Compare this query to the one in `03_duckdb_queries.ipynb`. **It is identical except for the path** — `lake/verkauf/...` became `s3://datalake/verkauf/...`. DuckDB (the compute/query engine) didn't change; only the storage layer underneath it did.

That's the separation of storage and compute in practice: the same engine can query a local folder, an S3 bucket, or (as in production) a cloud data lake — the SQL doesn't care.